# 03 - Validasi Silang (Cross Validation)
Materi Modul 3: validasi silang K-lipatan, cross_val_score, cross_val_predict, dan memilih model dengan CV.

Ide: data dibagi jadi **K lipatan (fold)**. Setiap fold bergantian jadi data uji, sisanya jadi data latih.
Hasilnya K skor -> rata-rata lebih stabil daripada satu kali split.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import KFold, cross_val_score, cross_val_predict, LeaveOneOut
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import PolynomialFeatures
from sklearn.pipeline import make_pipeline
from sklearn.metrics import r2_score

df = pd.read_csv('../datasets/diabetes.csv')
X = df.drop(columns='progres_penyakit')
y = df['progres_penyakit']

## 1. K-Fold manual (pakai loop)

In [ ]:
kf = KFold(n_splits=5, shuffle=True, random_state=42)

skor_manual = []
for i, (idx_train, idx_test) in enumerate(kf.split(X), start=1):
    model = LinearRegression()
    model.fit(X.iloc[idx_train], y.iloc[idx_train])
    skor = model.score(X.iloc[idx_test], y.iloc[idx_test])    # R2
    skor_manual.append(skor)
    print(f'Fold {i}: train={len(idx_train)} test={len(idx_test)} R2={skor:.4f}')

print('Rata-rata R2:', round(np.mean(skor_manual), 4))
print('Std R2      :', round(np.std(skor_manual), 4))

## 2. Visualisasi pembagian fold

In [ ]:
fig, ax = plt.subplots(figsize=(10, 3))
for i, (idx_train, idx_test) in enumerate(KFold(n_splits=5).split(X)):
    warna = np.zeros(len(X))
    warna[idx_test] = 1
    ax.scatter(range(len(X)), [i] * len(X), c=warna, cmap='coolwarm', marker='|', s=120)
ax.set_yticks(range(5)); ax.set_yticklabels([f'Fold {i+1}' for i in range(5)])
ax.set_xlabel('Indeks data'); ax.set_title('Merah = data uji, biru = data latih')
plt.show()

## 3. `cross_val_score` (cara singkat)

In [ ]:
skor = cross_val_score(LinearRegression(), X, y, cv=kf, scoring='r2')
print('R2 tiap fold:', skor.round(4))
print('Rata-rata   :', skor.mean().round(4))

# MSE di sklearn dibuat negatif (supaya "makin besar makin baik"), jadi dibalik tandanya
mse = -cross_val_score(LinearRegression(), X, y, cv=kf, scoring='neg_mean_squared_error')
print('RMSE rata-rata:', np.sqrt(mse).mean().round(2))

## 4. `cross_val_predict`
Mengembalikan prediksi untuk **setiap** data, dihasilkan saat data itu jadi data uji.

In [ ]:
pred_cv = cross_val_predict(LinearRegression(), X, y, cv=kf)
print('R2 dari prediksi CV:', round(r2_score(y, pred_cv), 4))

plt.scatter(y, pred_cv, alpha=0.5)
plt.plot([y.min(), y.max()], [y.min(), y.max()], 'r--')
plt.xlabel('Nilai asli'); plt.ylabel('Prediksi (CV)'); plt.title('Prediksi vs Asli')
plt.show()

## 5. Pengaruh jumlah fold (K)

In [ ]:
for k in [3, 5, 10, 20]:
    s = cross_val_score(LinearRegression(), X, y, cv=KFold(k, shuffle=True, random_state=42), scoring='r2')
    print(f'K={k:2d} | rata-rata R2={s.mean():.4f} | std={s.std():.4f}')

# Leave-One-Out: K = jumlah data (mahal untuk data besar)
loo = cross_val_score(LinearRegression(), X, y, cv=LeaveOneOut(), scoring='neg_mean_squared_error')
print('LOO RMSE:', round(np.sqrt(-loo.mean()), 2))

## 6. Kenapa `shuffle=True` penting?
Kalau data terurut (misal berdasarkan target), fold tanpa shuffle bisa sangat bias.

In [ ]:
df_urut = df.sort_values('progres_penyakit')
Xu, yu = df_urut.drop(columns='progres_penyakit'), df_urut['progres_penyakit']

tanpa = cross_val_score(LinearRegression(), Xu, yu, cv=KFold(5, shuffle=False), scoring='r2')
dengan = cross_val_score(LinearRegression(), Xu, yu, cv=KFold(5, shuffle=True, random_state=42), scoring='r2')
print('Tanpa shuffle:', tanpa.round(3))
print('Dengan shuffle:', dengan.round(3))

## 7. Memilih derajat polinomial dengan CV

In [ ]:
data = pd.read_csv('../datasets/kurva_sinus.csv')
Xs, ys = data[['x']], data['y']

derajat_list = range(1, 13)
rata = []
for derajat in derajat_list:
    model = make_pipeline(PolynomialFeatures(derajat), LinearRegression())
    s = -cross_val_score(model, Xs, ys, cv=KFold(5, shuffle=True, random_state=0), scoring='neg_mean_squared_error')
    rata.append(s.mean())

plt.plot(derajat_list, rata, marker='o')
plt.yscale('log'); plt.xlabel('Derajat polinomial'); plt.ylabel('MSE (CV)')
plt.title('Memilih derajat dengan validasi silang')
plt.show()
print('Derajat terbaik menurut CV:', list(derajat_list)[int(np.argmin(rata))])

## Catatan
- (kenapa CV lebih andal dari satu kali split, trade-off nilai K, kapan pakai LOO, dll.)